<a href="https://colab.research.google.com/github/Emersonsag6/SISTEMA-VACINA/blob/main/GERADOR_DE_ESCALA_BEL%C3%89M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Escalas Polo Base - Belém do Solimões
Versão ajustada:
- Seleção simples por checklist (sem dropdown por categoria)
- Gerenciamento de afastamentos com remoção via dropdown e limpar todos
- Destaque em tons de cinza para finais de semana e feriados no PDF
- Normalização de acentos, parsing robusto de datas, gravação atômica do histórico
"""
# ====================================================================
# Dependências (descomente / instale quando for rodar)
# ====================================================================
# !pip install -q gradio pandas openpyxl reportlab

# ====================================================================
# Importações e configurações
# ====================================================================
import os
import json
import unicodedata
import pandas as pd
from datetime import datetime, timedelta, date
import gradio as gr

from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

HISTORICO_FILE = "historico_rodizio.json"

# ====================================================================
# Histórico: carregar / salvar (escrita atômica)
# ====================================================================
def carregar_historico():
    if os.path.exists(HISTORICO_FILE):
        try:
            with open(HISTORICO_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {
        "Equipe 1": {},
        "Equipe 2": {},
        "MEDICOS_GLOBAL": 0
    }

def salvar_historico(historico):
    try:
        tmp = HISTORICO_FILE + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(historico, f, ensure_ascii=False, indent=4)
        os.replace(tmp, HISTORICO_FILE)
    except Exception as e:
        print(f"Erro ao salvar histórico: {e}")

# ====================================================================
# Utilitários: acentos, limpeza, datas
# ====================================================================
def remover_acentos(texto: str) -> str:
    if not isinstance(texto, str):
        return ""
    return ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))

def label_limpo(nome):
    if pd.isna(nome):
        return ""
    return str(nome).strip()

def converter_para_datetime(data):
    """Aceita str, date ou datetime; fallback: now()."""
    if data is None or (isinstance(data, str) and not str(data).strip()):
        return datetime.now()
    if isinstance(data, datetime):
        return data
    if isinstance(data, date):
        return datetime.combine(data, datetime.min.time())
    d_str = str(data).strip().split(" ")[0]
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(d_str, fmt)
        except ValueError:
            continue
    try:
        return pd.to_datetime(d_str, dayfirst=True).to_pydatetime()
    except Exception:
        return datetime.now()

# ====================================================================
# Função para adicionar feriados à lista
# ====================================================================
def adicionar_feriado_lista(data_feriado, lista_atual):
    if not data_feriado:
        return lista_atual
    dt_obj = converter_para_datetime(data_feriado)
    dt_fmt = dt_obj.strftime("%d/%m/%Y")
    linhas = [l.strip() for l in str(lista_atual or "").split("\n") if l.strip()]
    if dt_fmt not in linhas:
        linhas.append(dt_fmt)
    return "\n".join(linhas)

# ====================================================================
# Extração de profissionais (planilha) — com normalização de acentos
# ====================================================================
def extrair_profissionais_belem_avancado(file_path):
    try:
        excel_file = pd.ExcelFile(file_path)
        aba_alvo = next((s for s in excel_file.sheet_names if "belem" in s.lower()), excel_file.sheet_names[0])
        df_raw = pd.read_excel(file_path, sheet_name=aba_alvo)

        equipe_atual = "Equipe 1"
        profissionais = []

        mapeamento_cargos = {
            "ENFERMEIR": "Enfermeiro",
            "TÉCNICO DE ENFERMAGEM": "Téc. Enfermagem",
            "TECNICO DE ENFERMAGEM": "Téc. Enfermagem",
            "TÉC. ENF": "Téc. Enfermagem",
            "TEC. ENF": "Téc. Enfermagem",
            "TÉCNICO": "Téc. Enfermagem",
            "TECNICO": "Téc. Enfermagem",
            "CIRURGIÃO DENTISTA": "C. Dentista",
            "CIRURGIAO DENTISTA": "C. Dentista",
            "DENTISTA": "C. Dentista",
            "C. DENTISTA": "C. Dentista",
            "CD": "C. Dentista",
            "PSICOLOGO": "Psicólogo",
            "PSICÓLOGO": "Psicólogo",
            "FARMACEUTICO": "Farmacêutico",
            "FARMACÊUTICO": "Farmacêutico",
            "TECNICO DE LABORATORIO": "Téc. Laboratório",
            "TÉCNICO DE LABORATÓRIO": "Téc. Laboratório",
            "TEC DE LAB": "Téc. Laboratório",
            "TÉC. LAB": "Téc. Laboratório",
            "LABORATORIO": "Téc. Laboratório",
            "LABORATÓRIO": "Téc. Laboratório",
            "NUTRICIONISTA": "Nutricionista",
            "NUTRI": "Nutricionista",
            "ASSISTENTE SOCIAL": "Assistente Social",
            "ASSIST SOCIAL": "Assistente Social",
            "ASSIS SOCIAL": "Assistente Social",
            "MÉDICO": "Médico",
            "MEDICO": "Médico"
        }

        norm_map = {remover_acentos(k).upper(): v for k, v in mapeamento_cargos.items()}
        exclusoes = {remover_acentos(k).upper() for k in ["POLO", "ALDEIA", "UBSI", "BELÉM", "TRAB", "AREJ", "FÉRIAS", "FERIAS"]}

        for idx, row in df_raw.iterrows():
            row_str = " ".join(row.dropna().astype(str)).upper()

            if "EQUIPE 2" in row_str:
                equipe_atual = "Equipe 2"
                continue
            elif "EQUIPE 1" in row_str:
                equipe_atual = "Equipe 1"
                continue

            row_vals = [str(v).strip() for v in row.dropna().values if str(v).strip()]
            if len(row_vals) < 2:
                continue

            cargo_encontrado = ""
            nome_candidate = ""

            for val in row_vals:
                val_upper = str(val).upper()
                val_norm = remover_acentos(val_upper)

                for chave_norm, cargo_normalizado in norm_map.items():
                    if chave_norm in val_norm:
                        cargo_encontrado = cargo_normalizado
                        break

                if len(val.strip()) > 3 and not any(exc in val_norm for exc in exclusoes):
                    if not nome_candidate and not any(k in val_norm for k in norm_map.keys()):
                        nome_candidate = val

            if cargo_encontrado and nome_candidate:
                nome_limpo = label_limpo(nome_candidate)
                if cargo_encontrado == "Médico":
                    eq_vinculo = "Pool Geral (Médicos)"
                else:
                    eq_vinculo = equipe_atual

                profissionais.append({
                    "Equipe": eq_vinculo,
                    "Nome": nome_limpo,
                    "Cargo": cargo_encontrado,
                    "Identificador": f"{nome_limpo} ({cargo_encontrado})"
                })

        df = pd.DataFrame(profissionais).drop_duplicates()
        if df.empty:
            return pd.DataFrame(columns=["Equipe", "Nome", "Cargo", "Identificador"])
        return df
    except Exception as e:
        print(f"Erro na extração: {e}")
        return pd.DataFrame(columns=["Equipe", "Nome", "Cargo", "Identificador"])

# ====================================================================
# Rodízio com suporte a afastamentos
# ====================================================================
def montar_afastamentos_parsed(afastamentos):
    parsed = []
    if not afastamentos:
        return parsed
    for a in afastamentos:
        try:
            prof = a.get("prof")
            start = converter_para_datetime(a.get("start")).date()
            end = converter_para_datetime(a.get("end")).date()
            if prof and start and end:
                parsed.append({"prof": prof, "start": start, "end": end})
        except Exception:
            continue
    return parsed

def esta_ausente(prof_ident, dt_curr_date, afastamentos_parsed):
    for a in afastamentos_parsed:
        if a["prof"] == prof_ident and a["start"] <= dt_curr_date <= a["end"]:
            return True
    return False

def gerar_rodizio_12h(df_prof, equipe_sel,
                      profs_equipe_sel,
                      dt_inicio_str, dias_permanencia,
                      medicos_selecionados, dt_entrada_med_str, dt_saida_med_str,
                      dias_totais_escala, feriados_lista_str,
                      afastamentos=None):
    historico = carregar_historico()
    ponteiros_eq = historico.get(equipe_sel, {})
    idx_med_global = historico.get("MEDICOS_GLOBAL", 0)

    feriados_set = set()
    if feriados_lista_str:
        for f in str(feriados_lista_str).split("\n"):
            f_clean = f.strip()
            if f_clean:
                feriados_set.add(f_clean)

    profissionais_por_cargo = {}
    if profs_equipe_sel:
        for item in profs_equipe_sel:
            if "(" in item and ")" in item:
                cargo = item.split("(")[-1].replace(")", "").strip()
            else:
                parts = item.split()
                cargo = parts[-1] if parts else "Outro"
            profissionais_por_cargo.setdefault(cargo, []).append(item)

    if not medicos_selecionados:
        medicos_selecionados = ["Nenhum Médico em Área"]

    dt_inicio = converter_para_datetime(dt_inicio_str)

    try:
        dias_perm_val = int(float(dias_permanencia)) if dias_permanencia is not None else 20
    except (ValueError, TypeError):
        dias_perm_val = 20

    dt_fim_perm = dt_inicio + timedelta(days=dias_perm_val - 1)
    dt_entrada_med = converter_para_datetime(dt_entrada_med_str)
    dt_saida_med = converter_para_datetime(dt_saida_med_str)

    try:
        dias_totais_val = int(float(dias_totais_escala)) if dias_totais_escala is not None else 30
    except (ValueError, TypeError):
        dias_totais_val = 30

    if not profissionais_por_cargo:
        raise ValueError("Nenhum profissional por cargo foi selecionado para rodízio. Verifique a seleção da equipe.")

    indices_cargos = {cargo: ponteiros_eq.get(cargo, 0) for cargo in profissionais_por_cargo.keys()}
    afast_parsed = montar_afastamentos_parsed(afastamentos)

    escala = []
    dias_semana_pt = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

    for i in range(dias_totais_val):
        dt_curr = dt_inicio + timedelta(days=i)
        dia_str = dt_curr.strftime("%d/%m/%Y")
        sem_str = dias_semana_pt[dt_curr.weekday()]

        is_fds = dt_curr.weekday() >= 5
        is_feriado = dia_str in feriados_set
        requer_dois_turnos = is_fds or is_feriado

        turnos_do_dia = ["Diurno (12h)", "Noturno (12h)"] if requer_dois_turnos else ["Noturno (12h)"]

        for turno in turnos_do_dia:
            linha_escala = {
                "Data": dia_str,
                "Dia": sem_str,
                "Turno": turno,
            }

            for cargo, lista_profs in profissionais_por_cargo.items():
                disponiveis = [p for p in lista_profs if not esta_ausente(p, dt_curr.date(), afast_parsed)]
                if dt_inicio <= dt_curr <= dt_fim_perm:
                    if disponiveis:
                        idx = indices_cargos.get(cargo, 0)
                        p_escalado = disponiveis[idx % len(disponiveis)]
                        linha_escala[cargo] = p_escalado
                        indices_cargos[cargo] = idx + 1
                    else:
                        linha_escala[cargo] = "Sem Cobertura (Afastamento)"
                else:
                    linha_escala[cargo] = "Sem Cobertura (Fora da Permanência)"

            if dt_entrada_med <= dt_curr <= dt_saida_med:
                med_disponiveis = [m for m in medicos_selecionados if not esta_ausente(m, dt_curr.date(), afast_parsed)]
                if med_disponiveis:
                    p_med = med_disponiveis[idx_med_global % len(med_disponiveis)]
                    idx_med_global += 1
                else:
                    p_med = "Sem Médico Disponível (Afastamento)"
            else:
                p_med = "Fora do Período Médico"

            linha_escala["Médico(a)"] = p_med
            linha_escala["Observação"] = "Feriado" if is_feriado else ("Fim de Semana" if is_fds else "Dia Útil")

            escala.append(linha_escala)

    historico[equipe_sel] = indices_cargos
    historico["MEDICOS_GLOBAL"] = idx_med_global
    salvar_historico(historico)

    return pd.DataFrame(escala)

# ====================================================================
# Gerador de PDF com destaque para fins de semana e feriados
# ====================================================================
def gerar_pdf_escala(df_escala, equipe_nome, data_inicio_str):
    try:
        pdf_filename = f"Escala_Plantao_Belem_{equipe_nome.replace(' ', '_')}.pdf"
        doc = SimpleDocTemplate(
            pdf_filename,
            pagesize=landscape(A4),
            rightMargin=1.0*cm, leftMargin=1.0*cm, topMargin=1.0*cm, bottomMargin=1.0*cm
        )

        styles = getSampleStyleSheet()
        title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=13, leading=16, alignment=1, textColor=colors.HexColor("#003366"))
        subtitle_style = ParagraphStyle('SubTitleStyle', parent=styles['Normal'], fontSize=9, leading=11, alignment=1, textColor=colors.HexColor("#333333"))
        cell_style = ParagraphStyle('CellStyle', parent=styles['Normal'], fontSize=7, leading=9, alignment=1)
        cell_header = ParagraphStyle('CellHeader', parent=styles['Normal'], fontSize=7.5, leading=9, alignment=1, textColor=colors.white, fontName="Helvetica-Bold")

        elements = []
        elements.append(Paragraph("<b>DSEI ALTO RIO SOLIMÕES - POLO BASE DE BELÉM DO SOLIMÕES</b>", title_style))
        elements.append(Spacer(1, 3))
        elements.append(Paragraph(f"<b>ESCALA MULTIPROFISSIONAL DE PLANTÕES</b> | {equipe_nome.upper()} | Início: {data_inicio_str}", subtitle_style))
        elements.append(Spacer(1, 8))

        if isinstance(df_escala, (list, dict)):
            try:
                df_escala = pd.DataFrame(df_escala)
            except Exception:
                pass

        headers = list(df_escala.columns)
        table_data = [[Paragraph(h, cell_header) for h in headers]]

        for _, row in df_escala.iterrows():
            row_cells = [Paragraph(str(row[col]), cell_style) for col in headers]
            table_data.append(row_cells)

        num_cols = len(headers)
        largura_disponivel = 27.7 * cm
        col_width = largura_disponivel / num_cols if num_cols > 0 else largura_disponivel

        t = Table(table_data, colWidths=[col_width]*num_cols, repeatRows=1)

        style_cmds = [
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#003366")),
            ('ALIGN', (0,0), (-1,-1), 'CENTER'),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#CCCCCC")),
            ('BOTTOMPADDING', (0,0), (-1,-1), 2),
            ('TOPPADDING', (0,0), (-1,-1), 2),
        ]

        for row_idx in range(len(df_escala)):
            obs_val = ""
            try:
                obs_val = str(df_escala.iloc[row_idx]["Observação"])
            except Exception:
                obs_val = ""
            table_row_index = row_idx + 1
            if obs_val == "Feriado":
                style_cmds.append(('BACKGROUND', (0, table_row_index), (-1, table_row_index), colors.HexColor("#BBBBBB")))
            elif obs_val == "Fim de Semana":
                style_cmds.append(('BACKGROUND', (0, table_row_index), (-1, table_row_index), colors.HexColor("#DDDDDD")))

        t.setStyle(TableStyle(style_cmds))

        elements.append(t)
        elements.append(Spacer(1, 10))
        elements.append(Paragraph("____________________________________________________", subtitle_style))
        elements.append(Paragraph("<b>Responsável pela Escala / Chefia do Polo Base</b>", subtitle_style))

        doc.build(elements)
        return pdf_filename
    except Exception as e:
        raise gr.Error(f"Erro ao gerar PDF: {str(e)}")

# ====================================================================
# UI: helpers para cargos e profissionais por equipe (simplificados)
# ====================================================================
def obter_profissionais_por_equipe(df, equipe):
    if df is None or df.empty:
        return []
    subset = df[df["Equipe"] == equipe]
    if "Identificador" in subset.columns:
        return subset["Identificador"].tolist()
    return subset["Nome"].tolist()

# ====================================================================
# Afastamentos: adicionar, remover por índice, remover por seleção e limpar todos
# ====================================================================
def label_afastamento(idx, a):
    return f"{idx} - {a['prof']} ({a['start']} -> {a['end']})"

def adicionar_afastamento_ui(prof_ident, data_inicio_str, dias, lista_afastamentos):
    if not prof_ident:
        return lista_afastamentos or [], pd.DataFrame(columns=["prof", "start", "end", "dias"]), gr.update(choices=[])
    try:
        dias_int = int(float(dias))
        if dias_int <= 0:
            dias_int = 1
    except Exception:
        dias_int = 1

    start_dt = converter_para_datetime(data_inicio_str)
    end_dt = start_dt + timedelta(days=dias_int - 1)

    novo = list(lista_afastamentos or [])
    novo.append({
        "prof": prof_ident,
        "start": start_dt.strftime("%d/%m/%Y"),
        "end": end_dt.strftime("%d/%m/%Y"),
        "dias": dias_int
    })

    labels = [label_afastamento(i, a) for i, a in enumerate(novo)]
    return novo, pd.DataFrame(novo), gr.update(choices=labels, value=(labels[0] if labels else None))

def remover_afastamento_ui(index, lista_afastamentos):
    lst = list(lista_afastamentos or [])
    try:
        idx = int(index)
        if 0 <= idx < len(lst):
            lst.pop(idx)
    except Exception:
        pass
    labels = [label_afastamento(i, a) for i, a in enumerate(lst)]
    return lst, pd.DataFrame(lst), gr.update(choices=labels, value=(labels[0] if labels else None))

def remover_afastamento_selecionado_ui(selected_label, lista_afastamentos):
    lst = list(lista_afastamentos or [])
    if not selected_label:
        labels = [label_afastamento(i, a) for i, a in enumerate(lst)]
        return lst, pd.DataFrame(lst), gr.update(choices=labels, value=(labels[0] if labels else None))
    try:
        idx_str = str(selected_label).split(" - ")[0]
        idx = int(idx_str)
        if 0 <= idx < len(lst):
            lst.pop(idx)
    except Exception:
        pass
    labels = [label_afastamento(i, a) for i, a in enumerate(lst)]
    return lst, pd.DataFrame(lst), gr.update(choices=labels, value=(labels[0] if labels else None))

def limpar_todos_afastamentos_ui(lista_afastamentos):
    lst = []
    labels = []
    return lst, pd.DataFrame(lst), gr.update(choices=labels, value=None)

# ====================================================================
# Upload e geração da escala / PDF
# ====================================================================
def processar_upload(file):
    global df_profissionais_global
    if not file:
        eq1_list = obter_profissionais_por_equipe(df_profissionais_global, "Equipe 1")
        med_list = obter_profissionais_por_equipe(df_profissionais_global, "Pool Geral (Médicos)")
        return ("Nenhum arquivo enviado.", df_profissionais_global, gr.update(choices=eq1_list, value=eq1_list), gr.update(choices=med_list, value=med_list), gr.update(choices=eq1_list, value=None))
    file_path = None
    if hasattr(file, "name"):
        file_path = file.name
    elif isinstance(file, dict) and "name" in file:
        file_path = file["name"]
    else:
        try:
            file_path = file[0].name
        except Exception:
            return ("Formato de arquivo desconhecido pelo servidor.", df_profissionais_global, gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[]))

    novo_df = extrair_profissionais_belem_avancado(file_path)
    if novo_df is None or novo_df.empty:
        return ("Planilha processada, mas nenhum profissional foi encontrado.", df_profissionais_global, gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[]))

    df_profissionais_global = novo_df
    total = len(df_profissionais_global)
    eq1_list = obter_profissionais_por_equipe(df_profissionais_global, "Equipe 1")
    med_list = obter_profissionais_por_equipe(df_profissionais_global, "Pool Geral (Médicos)")

    return (f"✅ Planilha lida com sucesso! {total} profissionais mapeados.",
            df_profissionais_global,
            gr.update(choices=eq1_list, value=eq1_list),
            gr.update(choices=med_list, value=med_list),
            gr.update(choices=eq1_list, value=None)
           )

def executar_geracao_escala(equipe, profs_eq_sel, dt_enf, dias_enf, medicos_sel, dt_med_in, dt_med_out, dias_totais, feriados_str, lista_afastamentos):
    global df_profissionais_global

    try:
        df_escala = gerar_rodizio_12h(
            df_profissionais_global, equipe,
            profs_eq_sel,
            dt_enf, dias_enf,
            medicos_sel, dt_med_in, dt_med_out,
            dias_totais, feriados_str,
            afastamentos=lista_afastamentos
        )
        return df_escala
    except Exception as e:
        raise gr.Error(f"Erro ao gerar escala: {str(e)}")

def exportar_pdf(df_escala_editada, equipe, dt_enf):
    if df_escala_editada is None or len(df_escala_editada) == 0:
        raise gr.Error("Gere a escala antes de exportar para PDF.")
    return gerar_pdf_escala(df_escala_editada, equipe, dt_enf)

# ====================================================================
# Dados iniciais de exemplo
# ====================================================================
df_profissionais_global = pd.DataFrame([
    {"Equipe": "Equipe 1", "Nome": "Igor Azevedo", "Cargo": "Enfermeiro(a)", "Identificador": "Igor Azevedo (Enfermeiro(a))"},
    {"Equipe": "Equipe 1", "Nome": "João Souza", "Cargo": "Téc. Enfermagem", "Identificador": "João Souza (Téc. Enfermagem)"},
    {"Equipe": "Equipe 1", "Nome": "Dra. Ana Dentista", "Cargo": "C. Dentista", "Identificador": "Dra. Ana Dentista (C. Dentista)"},
    {"Equipe": "Equipe 1", "Nome": "Marcos Silva", "Cargo": "Téc. Laboratório", "Identificador": "Marcos Silva (Téc. Laboratório)"},
    {"Equipe": "Equipe 1", "Nome": "Carla Psicóloga", "Cargo": "Psicólogo", "Identificador": "Carla Psicóloga (Psicólogo)"},
    {"Equipe": "Equipe 1", "Nome": "Lucas Farmacêutico", "Cargo": "Farmacêutico", "Identificador": "Lucas Farmacêutico (Farmacêutico)"},
    {"Equipe": "Equipe 2", "Nome": "Crizane Silva", "Cargo": "Enfermeiro(a)", "Identificador": "Crizane Silva (Enfermeiro(a))"},
    {"Equipe": "Equipe 2", "Nome": "Ana Paula", "Cargo": "Téc. Enfermagem", "Identificador": "Ana Paula (Téc. Enfermagem)"},
    {"Equipe": "Equipe 2", "Nome": "Dr. Roberto Dentista", "Cargo": "C. Dentista", "Identificador": "Dr. Roberto Dentista (C. Dentista)"},
    {"Equipe": "Pool Geral (Médicos)", "Nome": "Dr. Carlos", "Cargo": "Médico", "Identificador": "Dr. Carlos (Médico)"},
    {"Equipe": "Pool Geral (Médicos)", "Nome": "Dra. Juliana", "Cargo": "Médico", "Identificador": "Dra. Juliana (Médico)"}
])

# ====================================================================
# Construção do App Gradio
# ====================================================================
eq1_iniciais = obter_profissionais_por_equipe(df_profissionais_global, "Equipe 1")
med_iniciais = obter_profissionais_por_equipe(df_profissionais_global, "Pool Geral (Médicos)")

with gr.Blocks(title="Escalas Polo Base - Belém do Solimões") as app:
    gr.Markdown("# 🏥 Sistema de Rodízio Multiprofissional de Plantões")
    gr.Markdown("### Polo Base de Belém do Solimões — Cobertura Completa de Saúde")

    with gr.Tab("1. Importar Planilha e Cadastros"):
        file_input = gr.File(label="Envie a planilha de escala (.xlsx)", file_types=[".xlsx", ".csv"])
        status_output = gr.Textbox(label="Status do Carregamento", value="Dados de Teste Carregados!", interactive=False)
        table_prof = gr.Dataframe(value=df_profissionais_global, label="Todos os Profissionais Encontrados na Planilha", interactive=False)

    with gr.Tab("2. Gerar Escala de Permanência"):
        with gr.Row():
            equipe_sel = gr.Radio(["Equipe 1", "Equipe 2"], label="1. Selecione a Equipe Multiprofissional em Área", value="Equipe 1")

        with gr.Row():
            with gr.Column():
                gr.Markdown("#### 👥 Profissionais em Área (Checklist)")
                prof_equipe_check = gr.CheckboxGroup(
                    label="Selecione quem está em área nesta escala (Enfermeiros, Dentistas, Psicólogos, Téc. Lab, etc.)",
                    choices=eq1_iniciais,
                    value=eq1_iniciais
                )
                btn_clear_check = gr.Button("Limpar checklist principal", variant="danger")
                dt_enf = gr.Textbox(label="Data de Entrada da Equipe (DD/MM/AAAA)", value=datetime.now().strftime("%d/%m/%Y"))
                dias_enf = gr.Number(label="Dias de Permanência da Equipe", value=20)
                dias_totais = gr.Number(label="Total de Dias da Tabela Exibida", value=30)

            with gr.Column():
                gr.Markdown("#### 👨‍⚕️ Cobertura dos Médicos em Área")
                medicos_sel = gr.CheckboxGroup(
                    label="Médicos Escalados no Polo Base",
                    choices=med_iniciais,
                    value=med_iniciais
                )
                dt_med_in = gr.Textbox(label="Data Entrada Médico(s) (DD/MM/AAAA)", value=datetime.now().strftime("%d/%m/%Y"))
                dt_med_out = gr.Textbox(label="Data Saída Médico(s) (DD/MM/AAAA)", value=(datetime.now() + timedelta(days=30)).strftime("%d/%m/%Y"))

        equipe_sel.change(lambda equipe: (gr.update(choices=obter_profissionais_por_equipe(df_profissionais_global, equipe), value=obter_profissionais_por_equipe(df_profissionais_global, equipe)),
                                         gr.update(choices=obter_profissionais_por_equipe(df_profissionais_global, equipe), value=None)),
                          inputs=[equipe_sel], outputs=[prof_equipe_check, gr.State(value=None)])

        with gr.Row():
            with gr.Column():
                gr.Markdown("#### 📅 Adicionar Feriados à Escala")
                cal_feriado = gr.Textbox(
                    label="Data do Feriado (Sugerido: DD/MM/AAAA)",
                    value=datetime.now().strftime("%d/%m/%Y"),
                    placeholder="DD/MM/AAAA"
                )
                btn_add_feriado = gr.Button("➕ Adicionar Feriado à Lista", variant="secondary")

            with gr.Column():
                feriados_txt = gr.Textbox(
                    label="Feriados Cadastrados (DD/MM/AAAA)",
                    lines=4,
                    value="",
                    placeholder="Exemplo:\n07/09/2026\n12/10/2026"
                )

        btn_add_feriado.click(adicionar_feriado_lista, inputs=[cal_feriado, feriados_txt], outputs=[feriados_txt])

        # Afastamentos UI
        gr.Markdown("#### 🚫 Afastamentos / Licenças / Atestados")
        prof_para_afastar = gr.Dropdown(label="Profissional para afastamento", choices=eq1_iniciais, value=None)
        data_inicio_afast = gr.Textbox(label="Data Início (DD/MM/AAAA)", value=datetime.now().strftime("%d/%m/%Y"))
        dias_afast = gr.Number(label="Dias de Afastamento", value=1)
        btn_add_afast = gr.Button("Adicionar Afastamento", variant="secondary")
        tabela_afast = gr.Dataframe(label="Afastamentos cadastrados", interactive=False)
        input_index_remover = gr.Number(label="Índice a remover (0-based) — alternativa", value=None)
        btn_remover_afast = gr.Button("Remover Afastamento (por índice)", variant="danger")

        gr.Markdown("Ou remova selecionando um afastamento:")
        afastamentos_dropdown = gr.Dropdown(label="Afastamentos existentes", choices=[], value=None)
        btn_remover_afast_selec = gr.Button("Remover afastamento selecionado", variant="danger")
        btn_limpar_tudo_afast = gr.Button("Limpar todos afastamentos", variant="danger")

        afastamentos_state = gr.State(value=[])

        btn_add_afast.click(adicionar_afastamento_ui, inputs=[prof_para_afastar, data_inicio_afast, dias_afast, afastamentos_state], outputs=[afastamentos_state, tabela_afast, afastamentos_dropdown])
        btn_remover_afast.click(remover_afastamento_ui, inputs=[input_index_remover, afastamentos_state], outputs=[afastamentos_state, tabela_afast, afastamentos_dropdown])
        btn_remover_afast_selec.click(remover_afastamento_selecionado_ui, inputs=[afastamentos_dropdown, afastamentos_state], outputs=[afastamentos_state, tabela_afast, afastamentos_dropdown])
        btn_limpar_tudo_afast.click(limpar_todos_afastamentos_ui, inputs=[afastamentos_state], outputs=[afastamentos_state, tabela_afast, afastamentos_dropdown])

        # Botão gerar escala e exibição
        btn_gerar = gr.Button("⚡ Gerar Rodízio Multiprofissional", variant="primary")
        gr.Markdown("### 📝 Escala Gerada por Cargo e Data (Editável)")
        table_escala = gr.Dataframe(label="Colunas geradas automaticamente conforme as categorias dos profissionais selecionados.", interactive=True)

        btn_gerar.click(
            executar_geracao_escala,
            inputs=[equipe_sel, prof_equipe_check, dt_enf, dias_enf, medicos_sel, dt_med_in, dt_med_out, dias_totais, feriados_txt, afastamentos_state],
            outputs=[table_escala]
        )

        file_input.change(processar_upload, inputs=[file_input], outputs=[status_output, table_prof, prof_equipe_check, medicos_sel, prof_para_afastar])

        with gr.Row():
            btn_pdf = gr.Button("📄 Gerar PDF para Impressão", variant="secondary")
            pdf_file_output = gr.File(label="Download do PDF Pronto")

        btn_pdf.click(
            exportar_pdf,
            inputs=[table_escala, equipe_sel, dt_enf],
            outputs=[pdf_file_output]
        )

app.launch(theme=gr.themes.Soft(), share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5b5cf2b8ce16cec66c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
